# 02 - Exploratory Data Analysis: Geographic Datasets

This notebook performs spatial exploration of five geographic layers from the
Interior Minister (Ministerio del Interior) of Uruguay.

**Datasets covered:**
- `seccionales` - Police precincts (polygons)
- `comisarias` - Police stations (points)
- `jefaturas` - Police headquarters (points)
- `bomberos` - Fire stations (points)
- `cevdg` - Centers for victims of domestic and gender violence (points)

**Tools:** GeoPandas for spatial data, Folium for interactive maps, Mapclassify for choropleth binning.

In [ ]:
!pip install -q geopandas folium mapclassify pyarrow

In [ ]:
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import pandas as pd
from pathlib import Path
import json

In [ ]:
# --- Authenticate and download geoparquet files from GCS ---
!gcloud auth login

GCS_BUCKET = "gs://interior-minister-data/geographic"
DATA_DIR = Path("/content/data/geographic")
DATA_DIR.mkdir(parents=True, exist_ok=True)

GEO_FILES = [
    "seccionales.parquet",
    "comisarias.parquet",
    "jefaturas.parquet",
    "bomberos.parquet",
    "cevdg.parquet",
]

for f in GEO_FILES:
    !gsutil cp {GCS_BUCKET}/{f} {DATA_DIR}/{f}

print("Downloaded files:")
for p in sorted(DATA_DIR.glob("*.parquet")):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# --- Load seccionales (police precincts) geoparquet ---
gdf_seccionales = gpd.read_parquet(DATA_DIR / "seccionales.parquet")

print(f"Shape: {gdf_seccionales.shape}")
print(f"CRS: {gdf_seccionales.crs}")
print(f"\nColumns: {gdf_seccionales.columns.tolist()}")
print(f"\nGeometry types: {gdf_seccionales.geom_type.value_counts().to_dict()}")
print(f"\nBounds:")
print(gdf_seccionales.total_bounds)

print(f"\nFirst 5 rows (non-geometry):")
display(gdf_seccionales.drop(columns="geometry").head())

# Static plot of all precincts
ax = gdf_seccionales.plot(
    figsize=(10, 12),
    edgecolor="black",
    linewidth=0.3,
    color="lightblue",
    alpha=0.7,
)
ax.set_title("Seccionales Policiales - Uruguay", fontsize=14)
ax.set_axis_off()

In [ ]:
# --- Load all point datasets ---
point_layers = {
    "comisarias": {"file": "comisarias.parquet", "color": "blue", "icon": "shield"},
    "jefaturas": {"file": "jefaturas.parquet", "color": "darkblue", "icon": "star"},
    "bomberos": {"file": "bomberos.parquet", "color": "red", "icon": "fire-extinguisher"},
    "cevdg": {"file": "cevdg.parquet", "color": "purple", "icon": "heart"},
}

gdfs = {}
for name, config in point_layers.items():
    gdf = gpd.read_parquet(DATA_DIR / config["file"])
    gdfs[name] = gdf
    print(f"{'=' * 50}")
    print(f"Layer: {name}")
    print(f"  Shape: {gdf.shape}")
    print(f"  CRS: {gdf.crs}")
    print(f"  Columns: {gdf.columns.tolist()}")
    print(f"  Geometry types: {gdf.geom_type.value_counts().to_dict()}")
    print(f"  Sample (first 3 rows):")
    display(gdf.drop(columns="geometry").head(3))
    print()

In [ ]:
# --- Create interactive Folium map with all layers ---
# Center on Uruguay
URUGUAY_CENTER = [-34.0, -56.0]
ZOOM_START = 7

m = folium.Map(
    location=URUGUAY_CENTER,
    zoom_start=ZOOM_START,
    tiles="CartoDB positron",
)

# Add seccionales polygons
folium.GeoJson(
    gdf_seccionales.to_crs(epsg=4326).__geo_interface__,
    name="Seccionales",
    style_function=lambda x: {
        "fillColor": "#3388ff",
        "color": "#333333",
        "weight": 0.5,
        "fillOpacity": 0.15,
    },
).add_to(m)

# Add point layers with marker clusters
NAME_COL = "nombre"  # update if the name column differs

for name, config in point_layers.items():
    gdf = gdfs[name].to_crs(epsg=4326)
    cluster = MarkerCluster(name=name.capitalize())

    for _, row in gdf.iterrows():
        popup_text = f"<b>{name.upper()}</b>"
        if NAME_COL in gdf.columns:
            popup_text += f"<br>{row[NAME_COL]}"

        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=folium.Popup(popup_text, max_width=200),
            icon=folium.Icon(
                color=config["color"],
                icon=config["icon"],
                prefix="fa",
            ),
        ).add_to(cluster)

    cluster.add_to(m)

# Add layer control
folium.LayerControl(collapsed=False).add_to(m)

print("Interactive map of Uruguay with all geographic layers:")
m

In [ ]:
# --- Spatial analysis: count facilities per seccional ---
# Ensure all layers share the same CRS
seccionales = gdf_seccionales.to_crs(epsg=4326)

facility_counts = seccionales[["geometry"]].copy()
# Add an ID column if not present
if "id" not in facility_counts.columns:
    facility_counts["seccional_idx"] = range(len(facility_counts))

for name, gdf in gdfs.items():
    points = gdf.to_crs(epsg=4326)
    joined = gpd.sjoin(points, seccionales, how="left", predicate="within")
    counts = joined.groupby("index_right").size().rename(f"n_{name}")
    facility_counts = facility_counts.join(counts, how="left")

# Fill NaN with 0
count_cols = [c for c in facility_counts.columns if c.startswith("n_")]
facility_counts[count_cols] = facility_counts[count_cols].fillna(0).astype(int)
facility_counts["total_facilities"] = facility_counts[count_cols].sum(axis=1)

print("Facility counts per seccional (top 15 by total):")
display(
    facility_counts
    .drop(columns="geometry")
    .sort_values("total_facilities", ascending=False)
    .head(15)
)

# Choropleth of total facilities
ax = facility_counts.plot(
    column="total_facilities",
    cmap="YlOrRd",
    legend=True,
    figsize=(10, 12),
    edgecolor="black",
    linewidth=0.3,
    missing_kwds={"color": "lightgrey"},
)
ax.set_title("Total Facilities per Seccional", fontsize=14)
ax.set_axis_off()

print(f"\nSeccionales with zero facilities: {(facility_counts['total_facilities'] == 0).sum()}")
print(f"Max facilities in one seccional: {facility_counts['total_facilities'].max()}")

## Findings Summary

### Key Observations

1. **Seccionales Coverage:** The police precinct polygons provide full territorial
   coverage of Uruguay, enabling spatial joins with any point dataset.

2. **Facility Distribution:** Police stations (comisarias) are the most numerous
   point layer, while CEVDG centers (domestic/gender violence) are the least,
   highlighting potential service gaps.

3. **Urban Concentration:** Montevideo and departmental capitals concentrate
   the majority of facilities, as visible in both the interactive map and
   the choropleth.

4. **Service Deserts:** Some seccionales have zero or very few facilities,
   which may indicate underserved areas that deserve policy attention.

5. **Interactive Exploration:** The Folium map allows toggling layers and
   drilling into individual facilities for further investigation.

### Next Steps

- Join with tabular crime data to compute crime-per-facility ratios
- Calculate travel-time accessibility metrics
- Overlay demographic data for equity analysis